# [LOG04 미션 03] 관리 물자 고갈 위험 등급 및 긴급 재보급 요청 승인/보류 기준서 초안 작성 — 교수자용 해설판 (Workbook)

- **산출물 경로**: `analysis/log04_uncertainty_assessment.md`
- **실습 환경**: Python 3.12 표준 라이브러리 (`pathlib`, `csv`) — 별도 라이브러리 설치 및 인터넷 연결 불필요

> **실습·보안 안내**: 제공된 교육용 가상자료만 사용하며 실제 조직명·물자 제원·재고·개인정보는 입력하지 마세요. 점수나 등급으로 승인을 자동 결정하지 않으며, 최종 판단은 반드시 사람이 내립니다.

---

## 미션 3. 관리 물자 고갈 위험 등급 및 긴급 재보급 요청 승인/보류 기준서 초안 작성

### 목표

이번 미션에서는 교육장에서 제공하는 생성형 AI와 함께 「부족 위험과 추가 확인을 담은 기준서」 초안을 만듭니다.

물자가 부족할 것이라는 말은 남은 양과 도착 시간을 함께 봐야 이해할 수 있습니다.

자료가 맞지 않거나 비어 있을 때 무엇을 확인할지도 함께 적어야 문서를 쓸 수 있습니다.

준비할 자료는 다음과 같습니다: K1~K4, Q1~Q6, 미션 1·2의 계산과 기준. AI가 제안한 초안을 받은 뒤 사실과 숫자, 빠진 항목, 과장된 표현을 확인하고 직접 고쳐 씁니다.

끝나면 완성한 문서와 자료 대조 기록, 사용한 프롬프트, 고친 이유가 남습니다.

문장마다 근거를 남겨 다음 미션에서도 같은 자료를 찾아 쓸 수 있게 합니다.

수량은 단위와 기간을 함께 적고, 계산 결과는 계산기로 다시 확인합니다.

자료에 없는 값은 추측하지 않고 미확인으로 남깁니다.

다른 담당자가 문장과 표를 읽고 같은 자료를 찾을 수 있게 근거를 붙입니다.

### 상황과 자료

A팀이 받은 요청 Q1~Q6과 기록 쌍 K1~K4의 수량·시각·단위가 서로 다릅니다.

B팀은 어떤 요청을 검토하고 어떤 요청을 더 확인할지 설명이 필요합니다.

**준비할 자료**: K1~K4, Q1~Q6, 미션 1·2의 계산과 기준

**뜻 알아두기**: 집계 범위는 어느 거점·품목·기간을 합친 값인지 뜻합니다.

단위가 다른 수량은 더하지 않습니다.

정상·주의·경계·위험 조건은 자료에 근거해 정하고, 확인되지 않은 값은 미확인으로 남깁니다.


In [ ]:
# [환경 준비] 미션 3 실습 데이터셋 및 빈 산출물 서식 생성 (기존 파일은 보존)
from pathlib import Path

payload = {'02_데이터셋/거점_보유소모_대장.csv': "[교육용가상자료] 거점 보유·소모 대장 — 가상 조직 A팀 3개 거점의 D-3일~D일 기록. 수량 단위는 품목별(발·세트·묶음). 실제 재고·제원과 무관하다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n거점,품목,단위,D-3일보유량,D-2일소모,D-1일소모,D일소모,입고량_D-2~D,D일보유량_보고,일평균소모량_보고,리드타임_일,보고출처,보고시각,비고\n북곡,AM-1,발,42,6,6,6,0,24,6,3,창고 실사+서명,D일 17:00,D-3일 소모는 3발(이전 수준)\n북곡,DP-2,세트,60,5,5,5,0,40,5,2,불출 대장 집계,D일 17:00,파손 손실 기록란 비어 있음\n북곡,PG-3,묶음,150,10,10,10,0,120,10,1,창고 실사,D일 17:00,\n서림,AM-1,발,27,3,3,3,0,18,3,3,담당자 A 대장 집계,D일 18:00,담당자 B는 소모 5/일로 보고(K4)\n서림,DP-2,세트,24,6,6,0,0,12,4,2,불출 대장 집계,D일 18:00,D일 소모는 18시 집계 전(0으로 기재). 파손 손실과 정상 소모가 한 칸에 합산\n서림,PG-3,묶음,75,15,15,15,0,30,15,1,담당자 구두,D일 18:00,수량 '30 정도'의 어림값\n해솔,AM-1,발,20,4,4,4,0,8,4,3,전일 보고서 인용,D-1일 18:00,전일 인용은 D-1일 18시 출처 시각이다. D일 19시 대장 담당자가 D일 소모 4를 계산용 추정 기입하고 보유량 8을 산출했다. D일 실사값 아님. 현재량 확인 후 사용\n해솔,DP-2,세트,26,2,2,2,0,20,2,2,창고 실사,D일 16:00,\n해솔,PG-3,묶음,78,6,6,6,0,60,6,1,창고 실사,D일 16:00,야간 소모량 미반영\n", '02_데이터셋/재보급_요청대장.csv': "[교육용가상자료] 재보급 요청대장 — D일 가상 조직 A팀이 지원팀에 올린 요청. '요청서기재_보유일수'는 요청 거점이 스스로 적은 값이며 대장 재계산과 다를 수 있다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n요청ID,거점,품목,요청수량,요청시각,요청사유,요청서기재_보유일수,근거유형,수송로상태,도착예정,비고\nQ4,해솔,AM-1,10,D일 07:30,보유일수 2일로 리드타임 3일보다 짧음,2.0,전일 보고서 인용,양호,D+3일,R3와 같은 자료\nQ1,북곡,AM-1,12,D일 09:00,현장 대응 빈도 증가로 소모 6발/일,4.0,창고 실사+서명,양호,D+3일,R1과 같은 자료\nQ3,해솔,PG-3,60,D일 11:00,예방,4.0,요청서 1장(실사·대장 없음),양호,D+1일,C3와 같은 요청. 사유가 '예방'뿐\nQ5,서림,PG-3,30,D일 14:00,일 소모 15묶음,2.0,담당자 구두,양호,D+1일,R4와 같은 자료\nQ6,북곡,DP-2,20,D일 16:00,예방,8.0,근거 없음,양호,D+2일,B4와 같은 요청\nQ2,서림,DP-2,16,D일 18:30,이틀 만에 절반 소진(파손 포함 여부 불명),3.0,불출 대장 집계,지연 1일,D+3일,R2·C2와 같은 자료\n", '02_데이터셋/20261019b_선택미션_대체입력.md': '# [교육용가상자료] 선택 미션 미수행 시 제공 대체입력\n\n필수 미션 1·미션 2·미션 3·미션 5·미션 7·미션 9·미션 10의 Q1·Q2·Q3 모두 PASS일 때 최종 PASS다. 선택 미션 4·미션 6·미션 8은 산출물과 Q1·Q2·Q3 모두 가점/참고이며 최종 PASS에서 제외합니다. 선택 미션 미수행 시 02_데이터셋/20261019b_선택미션_대체입력.md의 제공 기본값·빈 서식을 사용한다. 후속 필수 미션에서 그 입력을 적용한 증거는 평가하지만 선택 미션 수행물·AI 세션을 소급 요구하지 않는다.\n\n제공 기본값은 정답도, 학습자 수행물도 아니다. 후속 미션에서 사용한 버전·값·자료 근거를 표시한다.\n\n## 미션 4 대체 규칙 v1\n\n제공 근거 가중치: 보유일수<리드타임 +3, 직전 2일보다 당일 소모 증가 +2, G4·G5·G6 -3, 근거 없는 예방 요청 -2. 모르는 값은?로 별도 기록한다. 종합 점수 = 상향 근거 가중치의 합 − 하향 근거 가중치 절댓값의 합. 규칙1: 점수3 이상은 긴급 승인 검토 후보. 규칙2: G4 이상 또는 독립 교차확인 불일치·미확인이면 보류. 규칙3: 확인 불가 1개 이상이면 추가 확인 요청. 규칙4: 사람의 최종 결정·운영 규칙 우선. 규칙2·3 보류 조건은 점수보다 우선한다. 미션 4의 12항목 설계·S1~S3·레드팀 제출은 요구하지 않는다.\n\n## 미션 6 대체 빈 서식\n\n| 요청ID | 거점 | 품목 | 보유일수 | 여유 | 위험 등급 | 요청수량 | 판단·근거 |\n|---|---|---|---|---|---|---|---|\n| Q1 | | | | | | | |\n| Q2 | | | | | | | |\n| Q3 | | | | | | | |\n| Q4 | | | | | | | |\n| Q5 | | | | | | | |\n| Q6 | | | | | | | |\n\n미션 9에서 필요한 현황은 미션 1·미션 3 결과로 이 빈 서식을 채운다. 미션 6의 메모3종 변환·금지표현6개는 요구하지 않는다.\n\n## 미션 8 대체 기준 유지안 v1\n\n이전 기준=미션 2·미션 3·미션 5 보존본, 수정 기준=동일 기준 유지안 v1. 미션 9의 V1~V7에 각각 재적용하고 동일 결과는 근거·유지 이유로 기록한다. 제공된 과거 판단과 내 이전 기준은 분리한다. 미션 8 쟁점5건·반박·AI 세션 제출은 요구하지 않는다.\n\n## 미션 10 부록\n\n수행한 선택 미션은 실제 산출물을, 미수행한 선택 미션은 이 제공 기본값·빈 서식과 후속 필수 미션에서 작성한 적용 기록을 부록에 넣는다. 원본은 보존하고 기준 유지·변경 내역을 표시한다.\n\n## 사실·판단·조치·미확인 분류\n\n사실은 제공 자료에서 확인한 내용, 판단은 사실을 기준에 적용한 해석, 조치는 다음에 할 행동, 미확인은 근거가 부족한 항목입니다.\n\n| 항목 | 사실과 자료 ID | 판단과 적용 기준 | 조치 | 미확인·재확인 대상 |\n|---|---|---|---|---|\n| 직접 작성 | | | | |\n', '03_참고자료/20261019b_정오표.md': '# 실습 적용 기준 — 원문 보존 / 수정본 채점\n\n원문은 보존 원문 및 학습자용 보존본에 남아 있습니다. 원문 대조용 보존본은 유지합니다. 현재 실습에는 아래 표의 수정 값을 적용하세요.\n\n| 항목 | 원문 값/종전 값 | 수정 값 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 선택 미션 평가 | 선택 미션 Q1만 선택, 산출물·Q2·Q3 필수(이전 안내) | 미션 4·미션 6·미션 8 산출물·Q1·Q2·Q3 모두 가점/참고 | 최종 PASS 범위 충돌 해소 | 필수 7미션의 Q1·Q2·Q3만 최종 PASS |\n| G 척도 경계 | G 이하/이상 혼용; 규칙2=미션 2 값 | 보류G≥4 / 통과후보G≤3 및 독립 교차확인 일치 | 경계 G3 중첩 제거 | 미션 2 서식4·미션 4 규칙2·미션 5 분기2 |\n| 전문 검토 순서 | 분기1→2→3→4→5 | 번호 유지, 순서4→3→1→2→5 | 전문 검토를 위험·신뢰도보다 먼저 의뢰 | B3 기술검토 분기3, B5 법무검토 분기4; 승인 아님 |\n| 분할 입력 | 길이 제한 안내: 시작 문서7절 | 현재 수행 단계 안의 분할 입력 절차 | 없는 참조 제거 | 미션 2 AI 비교 |\n| 보고 시각·추정값 | 전일 보고 인용인데 D일 소모4·보유8 | 전일 출처 시각 + D일19시 계산용 추정기입 명시 | 실사 시각과 추정 계산 구분; 수치 불변 | 해솔AM-1은 현재량 확인 후 사용, G4 유지 |\n| 요청 수신처·요약행 | M-1 본부 요청 / 요약표4행 | Q2 지원팀 요청(후속상신 미확인) / Q1~Q6 6행 | 수신처·행 수 정합 | 미션 6 실습 자료 서식1 |\n| 치환 기록 중복 | 치환 자료 끝 문단2회 | 동일 지시1회 | 중복 제거 | 미션 10 치환 적용 기록 |\n| 도메인 예시 | .go.kr 예시에 교육장 제공 자료 | .or.kr 예시로 이동; 기관·발행자 확인 우선 | 도메인 형태와 신뢰도 혼동 방지 | 학습 안내 문서; 접속 확인 아님 |\n', '03_참고자료/20261019_보완정오표.md': '# 2026-10-19 보완 실습 적용 기준\n\n원문 계열 표·목표를 덮어쓰지 않고 별도 보완 실습본을 제공합니다. 아래 안내에 따라 대장과 용어를 해석하세요.\n\n| 항목 | 원문 값/종전 안내 | 수정 값/보완 안내 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 미션 9 재검증 | 원문 서식 2의 개선 전 판단은 제공표 값 | 원문 값 유지 + 별도 표에 내 이전 기준 재적용 판단·수정 기준 적용 판단 추가 | 과거 판단과 내 기준 개정 효과의 혼동 방지 | 미션 9 재검증 7건의 과정·근거; 원문 Q1 문장·제공표·미측정 과거 시간 불변 |\n| 미션 7 카드 범위 | 원문 목표·수행은 거점 교대 포함, T1~T6에는 기준선 포함·거점 교대 없음 | 원문 불변, 거점 교대는 제공 카드 학습 범위 밖이라고 명시 | 실제 카드로 수행한 범위만 보고 | 제공 T1~T6만 수행·채점; 추가 카드·평가 조건 없음 |\n| 미션 1 후보 검토 | AI 후보 전부 채택이면 보완 | 전체 후보의 근거 대조·유지 이유 없이 전부 채택하면 보완 | 폐기 0건 허용 원칙과 정합화 | 미션 1 수행 절차의 통과 기준·Q2와 파생 문서; 근거 있는 전부 채택 인정 |\n| 멈춘 분기의 뜻 | 아니오가 나와 보류로 빠지는 분기 번호 | 예/아니오와 관계없이 보류·추가 확인·전문 검토로 넘긴 지점의 분기 번호 | 예 경로 중단도 기록 가능해야 함 | 시작 용어표·미션 5·미션 10 기록 안내, 원문 트리 불변 |\n', '02_데이터셋/log04_m03_sample.md': "# LOG04 미션 3 실습 샘플\n\n> `02_데이터셋/LOG04_실습자료_수정실습본.md`의 미션 3 부분입니다. 전체 자료는 프로젝트 첨부 자료의 학습자배포 ZIP에 있습니다.\n\n**미션 3. 관리 물자 고갈 위험 등급 및 긴급 재보급 요청 승인/보류 기준서 초안 작성**\n\n가. 제공 자료 1 - 어긋난 가상 재고 기록 쌍 4건 [교육용가상자료]\n\n| **쌍** | **기록 A** | **기록 B** | **어긋난 지점** |\n| --- | --- | --- | --- |\n| **K1** | 서림 DP-2 보유 12세트(거점 일지, D일 18시) | 서림 DP-2 보유 20세트(지원팀 집계, D일 06시) | 시각 차이와 소모 반영 여부 |\n| **K2** | 북곡 PG-3 120묶음 | 북곡 PG-3 1,200개 | 단위 혼용(묶음/개) |\n| **K3** | 해솔 AM-1 8발(창고) | 해솔 AM-1 11발(창고+이동 중 3발) | 집계 범위 차이 |\n| **K4** | 서림 AM-1 소모 3/일(담당자 A) | 서림 AM-1 소모 5/일(담당자 B) | 보고자 차이, 기준 기간 미기재 |\n\n가. 제공 자료 2 - 고갈 위험 등급 틀(이름은 고정, 조건과 처리는 조별 작성)\n\n| **등급** | **조건(보유일수·리드타임 여유)** | **처리(승인·보류·추가 확인 요청)** | **책임 직책(가상)** |\n| --- | --- | --- | --- |\n| **정상** | | | |\n| **주의** | | | |\n| **경계** | | | |\n| **위험** | | | |\n\n나. 작성 서식 1 - 긴급 재보급 요청 승인/보류 기준서 초안\n\n| **승인 검토 조건(모두 충족)** | |\n| --- | --- |\n\n| **보류 조건(하나라도 해당)** | |\n| --- | --- |\n\n| **추가 확인 요청 조건** | |\n| --- | --- |\n\n| **기록해야 할 문서** | |\n| --- | --- |\n\n| **최종 결정자(가상 직책)** | |\n| --- | --- |\n\n나. 작성 서식 2 - 어긋난 기록 분류와 원인 후보\n\n| **쌍** | **분류(시점·단위·집계 범위·보고자)** | **원인 후보** | **확인 방법** | **책임 직책** |\n| --- | --- | --- | --- | --- |\n| **K1** | | | | |\n| **K2** | | | | |\n| **K3** | | | | |\n| **K4** | | | | |\n\n나. 작성 서식 3 - 자료 공백 요인과 불확실성 영향도(8개 이상)\n\n| **번호** | **공백 요인** | **영향도(높음·중간·낮음)** | **대체 확인 수단** |\n| --- | --- | --- | --- |\n| **1** | | | |\n| **2** | | | |\n| **3** | | | |\n| **4** | | | |\n| **5** | | | |\n| **6** | | | |\n| **7** | | | |\n| **8** | | | |\n| **9** | | | |\n\n나. 작성 서식 4 - 확신 표현 규칙: '충분하다'처럼 단정하는 표현 대신 '( ) 기준 보유일수 ( )일, 확인 단계 G( ), 확인되지 않은 것은 ( )'의 형식으로 쓴다.\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 위험 등급 4단계의 조건을 수치로 적고 등급별 처리를 연결했다 | ( ) |\n| 승인 검토·보류·추가 확인 조건을 구분했다 | ( ) |\n| 어긋난 기록 4건에 원인 후보와 확인 방법을 적었다 | ( ) |\n| 자료 공백 요인을 8개 이상 찾고 영향도와 대체 확인 수단을 적었다 | ( ) |\n| 판단의 한계를 함께 쓰는 표현 규칙을 정했다 | ( ) |\n", 'analysis/log04_uncertainty_assessment.md': '# 미션 3 analysis/log04_uncertainty_assessment.md(고갈 위험 등급·승인/보류 기준서 초안, 상충 기록 분류표, 자료 공백·불확실성 영향도 매트릭스, 판단 한계 문서) (필수)\n\n빈 작성 양식입니다. 실제 수행 후 작성하세요.\n\n## 산출물 본문\n\n(여기에 작성)\n\n## 자료 ID와 판단 근거\n\n(여기에 작성)\n\n## 협업·AI 기록과 선별 이유\n\n(여기에 작성)\n\n## 점검 결과와 미수행 항목\n\n(여기에 작성)\n'}
for name, content in payload.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8')
print(f'미션 3 자료 준비 완료 ({len(payload)}개 파일 확인, 기존 작성 파일 보존)')


### 할 일

1. 시점·단위·집계 범위·보고자를 나누어 어긋난 부분을 표시합니다.

2. AI에게 4단계 위험 조건·처리·책임과 확인할 내용을 담은 기준서 초안을 요청합니다.

3. 요청 수량과 보유일수를 검산하고 근거 없는 숫자·확신 표현을 고칩니다.

4. 공백 8개 이상에 영향과 대체 확인 수단을 적습니다.

5. 위험 기준·상충 분류표·자료 공백 표·판단 한계를 저장합니다.

평가의 자료 수·숫자·근거 조건은 같은 미션 문제지의 확인 포인트와 대조합니다.

**핵심 수행·평가 포인트**
- 요청대장(`02_데이터셋/재보급_요청대장.csv`) Q1~Q6의 보유일수를 소수 첫째 자리로 재계산하고, 기재값과 다르면 `Q#`와 열 이름을 적어 `'확인 후 사용'`으로 분류합니다.
- K2(`북곡 PG-3` 묶음/개)는 환산 기준이 자료에 없으므로 임의 환산하지 말고 `'확인 필요'`로 둡니다.
- 위험 등급 4단계 조건을 수치(여유 일수)로 정해 9행을 배정하고, 서식 1에 미션 2 출처 건수·G 상한·보고 경과 시간·최종 결정자(가상 직책) 및 **'점수·등급이 승인을 자동 결정하지 않는다'** 문구를 명시합니다.
- 공백 8개 이상에 출처 ID·열 이름·영향도·대체 확인 수단을 적고, 서식 4 예문 2개 이상과 문서 맨 위 `'뒤 미션 입력값'`을 기록합니다.


### AI와 함께 해보기

교육장에서 제공하는 생성형 AI에 아래 요청을 복사하고 자료를 채웁니다.

```text
너는 가상 조직의 문서 작성 도우미야.
아래 자료만 사용해 「부족 위험과 추가 확인을 담은 기준서」의 초안을 만들어 줘.
자료: K1~K4, Q1~Q6, 미션 1·2의 계산과 기준.
[여기에 해당 가상 자료와 내 기준을 붙여넣기]
형식: 위험 단계 | 수치 조건 | 처리 | 책임자. 사례 | 차이 | 원인 후보 | 확인 방법. 공백 | 영향 | 대체 확인.
조건: 자료 ID를 붙여 줘. 사실·숫자·시각을 만들지 말고 알 수 없으면 미확인으로 써 줘. 단정 대신 근거와 한계를 함께 적어 줘. 최종 판단은 내가 할게.
```

**확인할 것 3가지**

1. 사실·숫자: 자료 ID, 원문, 수량, 단위, 시각을 자료와 대조합니다.

2. 빠진 항목: 요청한 표의 열과 미확인·근거·책임 항목이 있는지 봅니다.

3. 과장 표현: 근거 없이 반드시·확실하다라고 쓴 문장을 찾습니다.

### 작성 서식

더블클릭해 빈칸을 채우고 `Shift+Enter`로 표시하세요. 별도 산출물 파일에도 옮겨 저장할 수 있습니다.

**미션 3. 부족 위험과 추가 확인을 담은 기준서 (필수)**

나. 작성 서식 1 - 긴급 재보급 요청 승인/보류 기준서 초안

| **승인 검토 조건(모두 충족)** | |
| --- | --- |

| **보류 조건(하나라도 해당)** | |
| --- | --- |

| **추가 확인 요청 조건** | |
| --- | --- |

| **기록해야 할 문서** | |
| --- | --- |

| **최종 결정자(가상 직책)** | |
| --- | --- |

나. 작성 서식 2 - 어긋난 기록 분류와 원인 후보

| **쌍** | **분류(시점·단위·집계 범위·보고자)** | **원인 후보** | **확인 방법** | **책임 직책** |
| --- | --- | --- | --- | --- |
| **K1** | | | | |
| **K2** | | | | |
| **K3** | | | | |
| **K4** | | | | |

나. 작성 서식 3 - 자료 공백 요인과 불확실성 영향도(8개 이상)

| **번호** | **공백 요인** | **영향도(높음·중간·낮음)** | **대체 확인 수단** |
| --- | --- | --- | --- |
| **1** | | | |
| **2** | | | |
| **3** | | | |
| **4** | | | |
| **5** | | | |
| **6** | | | |
| **7** | | | |
| **8** | | | |
| **9** | | | |

나. 작성 서식 4 - 확신 표현 규칙: '충분하다'처럼 단정하는 표현 대신 '( ) 기준 보유일수 ( )일, 확인 단계 G( ), 확인되지 않은 것은 ( )'의 형식으로 쓴다.

다. 점검표

| **점검 항목** | **확인** |
| --- | --- |
| 위험 등급 4단계의 조건을 수치로 적고 등급별 처리를 연결했다 | ( ) |
| 승인 검토·보류·추가 확인 조건을 구분했다 | ( ) |
| 어긋난 기록 4건에 원인 후보와 확인 방법을 적었다 | ( ) |
| 자료 공백 요인을 8개 이상 찾고 영향도와 대체 확인 수단을 적었다 | ( ) |
| 판단의 한계를 함께 쓰는 표현 규칙을 정했다 | ( ) |

### AI 대조·수정 기록 및 남길 기록

| 자료 ID | 초안 문장 | 고친 문장 | 고친 이유 |
|---|---|---|---|
| | | | |

**남길 기록**

1. 프롬프트 전문과 실제 응답(연속 5줄 이상, 짧으면 전문), 세션 구분(협업 시 역할·요청·독립 판단 원문)

2. 사실·숫자 대조 / 빠진 항목 / 과장 표현 확인

3. 전체 후보 대조 및 채택·폐기(유지) 이유

4. 점검 결과와 미수행 항목


In [ ]:
# [보조 도구] 미션 3 — 요청대장 Q1~Q6 보유일수 재계산 대조 (확인 후 사용 판별)
import csv
from pathlib import Path

with Path("02_데이터셋/거점_보유소모_대장.csv").open(encoding="utf-8") as f:
    next(f)
    ledger_map = {(r["거점"], r["품목"]): r for r in csv.DictReader(f)}

with Path("02_데이터셋/재보급_요청대장.csv").open(encoding="utf-8") as f:
    next(f)
    reqs = list(csv.DictReader(f))

print(f"{'ID':<4} {'거점':<4} {'품목':<5} | {'기재보유일수':>10} {'대장재계산':>9} {'차이':>6} | 비고")
print("-" * 72)
for q in reqs:
    lr = ledger_map[(q["거점"], q["품목"])]
    recalc = round(float(lr["D일보유량_보고"]) / float(lr["일평균소모량_보고"]), 1)
    stated = float(q["요청서기재_보유일수"])
    diff = round(stated - recalc, 1)
    flag = "확인 후 사용" if diff != 0 else "일치"
    print(f"{q['요청ID']:<4} {q['거점']:<4} {q['품목']:<5} | {stated:>10.1f} {recalc:>9.1f} {diff:>+6.1f} | {flag}")


## [교수자용 해설 및 모범 답안 — 미션 3]

### 1. 지도 포인트 (평가 핵심 및 오답 함정)
- **Q3 재계산 불일치 (`해솔 PG-3`)**: 요청서에는 `요청서기재_보유일수 = 4.0`으로 적혀 있으나 대장(`60 / 6`)으로는 `10.0일`입니다. **`Q3, 요청서기재_보유일수 열 — 확인 후 사용`**으로 분류해야 합니다.
- **K2 단위 환산 금지**: `북곡 PG-3 120묶음 vs 1,200개`에 대해 임의로 `1묶음=10개`라고 가정해 환산하면 FAIL/보완 대상입니다. `'확인 필요'`로 유지해야 합니다.
- **자동 승인 금지 문구**: 서식 1에 `'점수나 등급이 승인을 자동으로 결정하지 않으며 최종 결정은 사람이 내린다'`는 취지의 문구를 반드시 포함해야 합니다.

### 2. 모범 답안 — Q1~Q6 대조 및 K1~K4 분류

| 요청ID | 거점·품목 | 기재 보유일수 | 대장 재계산 | 판정 |
|---|---|---|---|---|
| Q1 | 북곡 AM-1 | 4.0 | 4.0 | 일치 |
| Q2 | 서림 DP-2 | 3.0 | 3.0 | 일치 |
| **Q3** | **해솔 PG-3** | **4.0** | **10.0** | **불일치 (`확인 후 사용`)** |
| Q4 | 해솔 AM-1 | 2.0 | 2.0 | 일치 |
| Q5 | 서림 PG-3 | 2.0 | 2.0 | 일치 |
| Q6 | 북곡 DP-2 | 8.0 | 8.0 | 일치 |

- **K1~K4 분류**: K1 `시점(D일 18시 vs 06시)` / K2 `단위(묶음 vs 개 — 환산 기준 확인 필요)` / K3 `집계 범위(창고 8발 vs 창고+이동 중 11발)` / K4 `보고자(담당자 A 3/일 vs 담당자 B 5/일, 기준 기간 확인)`.


In [ ]:
# [교수자용 검증 코드] 미션 3 — Q1~Q6 재계산 불일치(Q3) 및 K2 단위 원문 확인
import csv
from pathlib import Path

with Path("02_데이터셋/거점_보유소모_대장.csv").open(encoding="utf-8") as f:
    next(f)
    lmap = {(r["거점"], r["품목"]): r for r in csv.DictReader(f)}

with Path("02_데이터셋/재보급_요청대장.csv").open(encoding="utf-8") as f:
    next(f)
    reqs = list(csv.DictReader(f))

bad = [q["요청ID"] for q in reqs if round(float(lmap[(q["거점"], q["품목"])]["D일보유량_보고"]) / float(lmap[(q["거점"], q["품목"])]["일평균소모량_보고"]), 1) != float(q["요청서기재_보유일수"])]
assert bad == ["Q3"]
print("[M3 검증 PASS] 요청대장 6건 중 기재 보유일수 불일치 = Q3 (4.0 vs 10.0)")
